# Demo 4 · How sure are we? Bootstrap confidence intervals

In [notebook 01](01_validation_and_hyperparameter_search.ipynb), cross-validation gave us a
mean score and a standard deviation. Reporting "accuracy = 0.70" still hides a question:
**how much would that number change with different data?** A **confidence interval** answers
it with a range of plausible values.

For many quantities there is no simple formula for that range. The **bootstrap** gets one
by simulation: it builds many new datasets by resampling the data we have, recomputes the
quantity on each, and looks at how much it varies.

By the end, you will be able to:

- Explain what sampling **with replacement** does and what the **out-of-bag** examples are.
- Compute a 95% percentile confidence interval for a classifier's accuracy.
- Visualise the uncertainty of a regression line and compute a confidence interval for its slope.
- Compare a bootstrap interval with the classical formula from `scipy.stats.linregress`.

You will reuse this technique in exercise Q6 of
[03 · Titanic, end to end](03_titanic_end_to_end.ipynb).

## Setup

In [ ]:
# OPTIONAL: run once in a fresh Colab runtime; skip during local development.
INSTALL_COLAB_PACKAGES = False  # Set to True to install; reset to False afterward.

import sys
import subprocess

# Detect Colab; otherwise use the local .venv.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    print("Local environment: skipped. Use uv sync in your terminal.")
elif not INSTALL_COLAB_PACKAGES:
    print("Installation skipped. Set INSTALL_COLAB_PACKAGES = True if this is a fresh Colab runtime.")
else:
    # Install into the Python running these cells.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])
    packages = [
        "numpy>=1.26",
        "pandas>=2.2,<4",
        "matplotlib>=3.7",
        "scikit-learn>=1.4",
        "scipy>=1.11",
    ]
    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install",
                           "--python", sys.executable, *packages])
    print("Installation finished. Restart the session if Colab requests it.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import linregress

from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

### Where the data comes from
The CSV files ship with the repository in `demo_4/datasets/`. Locally, the next cell
walks up from the notebook's folder until it finds that directory, so no download is
needed. In Colab, the repository is not on disk, so the same files are read directly
from GitHub.

In [ ]:
from pathlib import Path

DATA_URL = "https://raw.githubusercontent.com/milarobotlearningcourse/data_science/main/demo_4/datasets/"

# Locally, find demo_4/datasets by walking up from the notebook folder.
ROOT, DATA_DIR = Path.cwd(), None
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "demo_4" / "datasets").is_dir():
        ROOT, DATA_DIR = folder, folder / "demo_4" / "datasets"
        break


def data_source(filename):
    """Return the local CSV path if the repository is available, otherwise its URL."""
    if DATA_DIR is not None:
        return DATA_DIR / filename
    return DATA_URL + filename


print("Reading data from:", DATA_DIR or DATA_URL)

## A. Sampling with replacement

A bootstrap sample has the **same size** as the original data, but rows are drawn **with
replacement**: some rows appear several times, others not at all. The rows that were not
drawn are called **out-of-bag** (OOB) rows.

`rng` is a NumPy random generator. Fixing its seed makes the whole notebook reproducible.

In [ ]:
rng = np.random.default_rng(0)

rows = np.arange(10)
sample = rng.choice(rows, size=len(rows), replace=True)
out_of_bag = np.setdiff1d(rows, sample)

print("bootstrap sample:", np.sort(sample))
print("out-of-bag rows: ", out_of_bag)

On average, a bootstrap sample contains about **63%** of the distinct rows; the remaining
37% are out-of-bag. Since the model never sees the OOB rows during training, they can serve
as a test set.

## B. A confidence interval for a classifier's accuracy

### The data: Pima Indians diabetes
768 patients, 8 medical measurements, and whether the patient developed diabetes within
five years (`Outcome`: 1 = yes). The file has no header row, so we give the column names
ourselves.

In [ ]:
columns = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome",
]
diabetes = pd.read_csv(data_source("diabetes.csv"), header=None, names=columns)
print(diabetes.shape)
display(diabetes.head())
print("share of positive cases:", round(diabetes["Outcome"].mean(), 3))

About 35% of patients are positive, so a model that always predicts "no diabetes" would
already reach about 65% accuracy. Keep that baseline in mind when reading the results.

### The bootstrap loop
For each of 1,000 repetitions:

1. draw a bootstrap sample of row positions (with replacement);
2. train a decision tree on those rows;
3. measure its accuracy on the out-of-bag rows.

This takes a few seconds.

In [ ]:
X = diabetes.drop(columns="Outcome").to_numpy()
y = diabetes["Outcome"].to_numpy()
n = len(y)

n_iterations = 1000
scores = []
for _ in range(n_iterations):
    train_idx = rng.choice(n, size=n, replace=True)
    oob_idx = np.setdiff1d(np.arange(n), train_idx)

    model = DecisionTreeClassifier(random_state=0)
    model.fit(X[train_idx], y[train_idx])
    scores.append(accuracy_score(y[oob_idx], model.predict(X[oob_idx])))

scores = np.array(scores)
print(f"mean OOB accuracy over {n_iterations} bootstrap samples: {scores.mean():.3f}")

### From 1,000 scores to an interval
The **percentile method** keeps the middle 95% of the bootstrap scores: we cut 2.5% from
each tail, i.e. we take the 2.5th and 97.5th percentiles.

In [ ]:
confidence = 95
tail = (100 - confidence) / 2  # percentage cut from each side

lower = np.percentile(scores, tail)
upper = np.percentile(scores, 100 - tail)
print(f"{confidence}% confidence interval for accuracy: [{lower:.1%}, {upper:.1%}]")

plt.hist(scores, bins=30, color="lightgray", edgecolor="gray")
plt.axvline(lower, linestyle="--", color="black", label=f"{tail}th percentile")
plt.axvline(upper, linestyle="-", color="black", label=f"{100 - tail}th percentile")
plt.xlabel("Out-of-bag accuracy")
plt.ylabel("Number of bootstrap samples")
plt.legend()
plt.show()

**Reading the result.** The interval runs from the mid 60s to the mid 70s: about ten
points wide. A different dataset of the same size could easily give a score several points
higher or lower, and the low end of the interval is no better than the 65% baseline. Two models whose scores differ by less than that width
should not be called "better" or "worse" without more evidence.

*Precisely:* this interval describes how much the accuracy of "a decision tree trained on
data like ours" varies. It accounts for both the training rows and the test rows changing.

## C. Uncertainty in a regression line

### The data: SAT score and university GPA
84 students: their SAT score (an American admission test) and their grade point average
(GPA) at university. Does a higher SAT predict a higher GPA, and how precisely do we know the
slope?

In [ ]:
sat_gpa = pd.read_csv(data_source("sat-gpa.csv"))
print(sat_gpa.shape)
display(sat_gpa.head())

X_sat = sat_gpa[["SAT"]].to_numpy()  # 2-D: one column, as scikit-learn expects
y_gpa = sat_gpa["GPA"].to_numpy()

We fit a line on 1,000 bootstrap samples, draw every line in light grey, and keep each slope.
The red line is the fit on the full data.

In [ ]:
n_iterations = 1000
slopes = []
plt.figure(figsize=(10, 6))

for _ in range(n_iterations):
    idx = rng.choice(len(y_gpa), size=len(y_gpa), replace=True)
    lr = LinearRegression().fit(X_sat[idx], y_gpa[idx])
    slopes.append(lr.coef_[0])
    plt.plot(X_sat, lr.predict(X_sat), color="grey", alpha=0.05, zorder=1)

full_fit = LinearRegression().fit(X_sat, y_gpa)
plt.scatter(X_sat, y_gpa, color="orange", zorder=3, label="students")
plt.plot(X_sat, full_fit.predict(X_sat), color="red", zorder=4, label="fit on all the data")
plt.xlabel("SAT score")
plt.ylabel("University GPA")
plt.legend()
plt.show()

slopes = np.array(slopes)
print(f"slope on all the data: {full_fit.coef_[0]:.5f} GPA points per SAT point")
print(f"intercept: {full_fit.intercept_:.3f}")

The grey band is narrow near the middle of the data and wider at the edges: lines can
pivot around the centre, so predictions for extreme SAT scores are less certain.

### A confidence interval for the slope
Same percentile method as before, applied to the 1,000 slopes.

In [ ]:
slope_low, slope_high = np.percentile(slopes, [2.5, 97.5])
print(f"bootstrap 95% CI for the slope: [{slope_low:.5f}, {slope_high:.5f}]")

plt.hist(slopes, bins=30, color="lightgray", edgecolor="gray")
plt.axvline(slope_low, linestyle="--", color="black")
plt.axvline(slope_high, linestyle="--", color="black")
plt.xlabel("Bootstrap slope")
plt.ylabel("Number of bootstrap samples")
plt.show()

### Comparison with the classical formula
For a straight line, statistics also offers a formula. `linregress` returns the slope, the
correlation `r`, a p-value for the hypothesis "the true slope is 0", and the **standard
error** of the slope. A 95% interval is roughly slope ± 1.96 × standard error.

In [ ]:
fit = linregress(sat_gpa["SAT"], sat_gpa["GPA"])

print(f"slope:          {fit.slope:.5f}")
print(f"correlation r:  {fit.rvalue:.3f}")
print(f"p-value:        {fit.pvalue:.2e}")
print(f"standard error: {fit.stderr:.5f}")
print(f"classical 95% CI: [{fit.slope - 1.96 * fit.stderr:.5f}, {fit.slope + 1.96 * fit.stderr:.5f}]")

The two intervals are close. The classical formula relies on assumptions (for example,
normally distributed errors with constant spread); the bootstrap only assumes that our
84 students are representative of the population. When a formula exists and its assumptions
hold, use it. When no formula exists (an accuracy, an F1 score, a median…), the bootstrap
still works.

The interval stays well away from 0 and the p-value is tiny: higher SAT scores are clearly
associated with higher GPAs, by roughly 0.0017 GPA points per SAT point (about 0.17 GPA
points per 100 SAT points).

### Your turn
Compute a **90%** bootstrap confidence interval for the **intercept** of the regression.
You will need to store `lr.intercept_` in a new bootstrap loop. Is the 90% interval wider or
narrower than a 95% interval would be? Why?

In [ ]:
# Your turn: bootstrap the intercept and compute a 90% interval.

<details>
<summary>Reference answer — open after trying</summary>

```python
intercepts = []
for _ in range(1000):
    idx = rng.choice(len(y_gpa), size=len(y_gpa), replace=True)
    lr = LinearRegression().fit(X_sat[idx], y_gpa[idx])
    intercepts.append(lr.intercept_)

low, high = np.percentile(intercepts, [5, 95])
print(f"90% CI for the intercept: [{low:.3f}, {high:.3f}]")
```

A 90% interval keeps the middle 90% of the values (cut 5% from each tail), so it is
**narrower** than a 95% interval: we accept being wrong more often in exchange for a tighter
range. The intercept is the predicted GPA for a SAT score of 0, far outside the data, which
is why its interval is wide and it has no practical meaning here.

</details>

## Takeaways

- The bootstrap resamples the data **with replacement** to simulate new datasets of the same size.
- About 37% of rows are **out-of-bag** in each sample and can serve as a test set.
- The **percentile method** turns many bootstrap values into a confidence interval.
- It works for any statistic, including ones without a formula, such as a classifier's accuracy.

**Next:** [03 · Titanic, end to end](03_titanic_end_to_end.ipynb) puts notebooks 01 and 02
together on a complete project.

**To go further:** the scikit-learn documentation of
[`resample`](https://scikit-learn.org/stable/modules/generated/sklearn.utils.resample.html),
a helper that draws bootstrap samples of arrays or DataFrames.